# 03 — Content Creation

Validate LLM-powered social media post generation:
- Load strategy + insights
- Generate Mastodon posts (max 500 chars)
- Generate Bluesky posts (max 300 chars)
- Test bilingual output (DE + EN)
- Manual quality review

In [1]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os
import json

load_dotenv('../.env')

IONOS_API_TOKEN = os.getenv('IONOS_API_TOKEN')
print(f'IONOS token loaded: {"yes" if IONOS_API_TOKEN else "NO"}')

IONOS token loaded: yes


## 1. Load State from Previous Notebooks

In [2]:
from agent.storage import LocalStorage
from agent.models import Strategy, Insights

store = LocalStorage(base_dir='../state')

insights_raw = store.read('insights.json')
analysis = store.read('llm_analysis.json')
strategy = Strategy()  # Use defaults

if insights_raw is None or analysis is None:
    print('ERROR: Run notebooks 01 and 02 first.')
else:
    print(f'Insights loaded: {insights_raw["website_analytics"]["visitors"]} visitors')
    print(f'Growth opportunities: {len(analysis.get("growth_opportunities", []))}')
    print(f'Best pages for social: {len(analysis.get("best_pages_for_social", []))}')

Insights loaded: 24 visitors
Growth opportunities: 3
Best pages for social: 3


## 2. Content Planning — Pick Pages to Promote

In [ ]:
from agent.llm_client import LLMClient

llm = LLMClient(api_token=IONOS_API_TOKEN)

# Build content plan from LLM analysis
pages_to_promote = analysis.get('best_pages_for_social', [])
if not pages_to_promote:
    # Fallback: use top pages from analytics
    pages_to_promote = [
        {'url': p['path'], 'title': p['path'], 'reason': f'{p["visitors"]} visitors'}
        for p in insights_raw['website_analytics']['top_pages'][:5]
    ]

print(f'Pages to promote ({len(pages_to_promote)}):')
for p in pages_to_promote:
    print(f'  - {p["url"]} — {p["reason"]}')

Pages to promote (3):


KeyError: 'path'

## 3. Generate Mastodon Post (EN)

In [ ]:
# Pick first page to promote
page = pages_to_promote[0]
page_url = page['url'] if page['url'].startswith('http') else f'https://fretchen.eu{page["url"]}'

mastodon_prompt = f"""Write a Mastodon post (max 500 characters) about this blog article:

URL: {page_url}?utm_source=mastodon&utm_campaign=growth-agent
Page: {page.get('title', page['url'])}
Why promote: {page['reason']}

Context about the blog: fretchen.eu covers political economy, game theory,
quantum computing, blockchain/Web3, and AI tools.

Requirements:
- Hook in the first line (question or bold claim)
- Mention one specific insight from the article topic
- Include the link
- Add 2-3 relevant hashtags
- Tone: technical but accessible, opinionated
- Language: English

Do NOT use emojis excessively. One is fine.
Return ONLY the post text, nothing else."""

result = llm.chat(
    messages=[
        {'role': 'system', 'content': 'You write engaging social media posts for a technical blog. Be concise and punchy.'},
        {'role': 'user', 'content': mastodon_prompt}
    ],
    temperature=0.8,
    max_tokens=300,
)

mastodon_en = result['content'].strip()
print(f'=== Mastodon Post (EN) [{len(mastodon_en)} chars] ===')
print(mastodon_en)
print(f'\n{"✅ OK" if len(mastodon_en) <= 500 else "❌ TOO LONG"} ({len(mastodon_en)}/500 chars)')

## 4. Generate Bluesky Post (EN)

In [ ]:
bluesky_prompt = f"""Write a Bluesky post (max 300 characters) about this blog article:

URL: {page_url}?utm_source=bluesky&utm_campaign=growth-agent
Page: {page.get('title', page['url'])}
Why promote: {page['reason']}

Requirements:
- Concise, punchy hook
- Include the link
- No hashtags (Bluesky culture)
- Tone: conversational, insightful
- Language: English

Return ONLY the post text, nothing else."""

result = llm.chat(
    messages=[
        {'role': 'system', 'content': 'You write engaging social media posts for a technical blog. Be concise and punchy.'},
        {'role': 'user', 'content': bluesky_prompt}
    ],
    temperature=0.8,
    max_tokens=200,
)

bluesky_en = result['content'].strip()
print(f'=== Bluesky Post (EN) [{len(bluesky_en)} chars] ===')
print(bluesky_en)
print(f'\n{"✅ OK" if len(bluesky_en) <= 300 else "❌ TOO LONG"} ({len(bluesky_en)}/300 chars)')

## 5. Generate German Variant (Mastodon)

In [ ]:
mastodon_de_prompt = f"""Schreibe einen Mastodon-Post (max 500 Zeichen) über diesen Blog-Artikel:

URL: {page_url}?utm_source=mastodon&utm_campaign=growth-agent
Seite: {page.get('title', page['url'])}
Warum bewerben: {page['reason']}

Kontext: fretchen.eu behandelt politische Ökonomie, Spieltheorie,
Quantencomputing, Blockchain/Web3 und AI-Tools.

Anforderungen:
- Hook im ersten Satz (Frage oder starke These)
- Ein konkretes Insight aus dem Thema erwähnen
- Link einbinden
- 2-3 relevante Hashtags
- Duzen, nicht Siezen
- Ton: technisch aber verständlich, meinungsstark

Gib NUR den Post-Text zurück, nichts anderes."""

result = llm.chat(
    messages=[
        {'role': 'system', 'content': 'Du schreibst ansprechende Social Media Posts für einen technischen Blog. Sei prägnant und pointiert.'},
        {'role': 'user', 'content': mastodon_de_prompt}
    ],
    temperature=0.8,
    max_tokens=300,
)

mastodon_de = result['content'].strip()
print(f'=== Mastodon Post (DE) [{len(mastodon_de)} chars] ===')
print(mastodon_de)
print(f'\n{"✅ OK" if len(mastodon_de) <= 500 else "❌ TOO LONG"} ({len(mastodon_de)}/500 chars)')

## 6. Save Drafts to Content Queue

In [ ]:
from datetime import datetime
from agent.models import Draft, ContentQueue
import hashlib

def make_draft_id(content: str) -> str:
    return f'draft_{hashlib.md5(content.encode()).hexdigest()[:8]}'

drafts = [
    Draft(
        id=make_draft_id(mastodon_en),
        channel='mastodon',
        language='en',
        content=mastodon_en,
        source_blog_post=page.get('title', page['url']),
        link=f'{page_url}?utm_source=mastodon&utm_campaign=growth-agent',
    ),
    Draft(
        id=make_draft_id(bluesky_en),
        channel='bluesky',
        language='en',
        content=bluesky_en,
        source_blog_post=page.get('title', page['url']),
        link=f'{page_url}?utm_source=bluesky&utm_campaign=growth-agent',
    ),
    Draft(
        id=make_draft_id(mastodon_de),
        channel='mastodon',
        language='de',
        content=mastodon_de,
        source_blog_post=page.get('title', page['url']),
        link=f'{page_url}?utm_source=mastodon&utm_campaign=growth-agent',
    ),
]

queue = ContentQueue(drafts=drafts)
store.write('content_queue.json', queue)

print(f'Saved {len(drafts)} drafts to content_queue.json')
for d in drafts:
    print(f'  - {d.id} | {d.channel} ({d.language}) | {len(d.content)} chars')

In [ ]:
llm.close()
print('Done — Content creation validated.')